# Test #1 (Final) & Hackathon — Machine Learning & Deep Learning

**Author:** Ahmed Nadir — M2 SIEC, Université de Ghardaïa  
**Date:** 11-10-2026  
**Tools:** Scikit-learn · TensorFlow/Keras · Matplotlib (+ imbalanced-learn for SMOTE)

---

### What is inside
| Part | Subject | Models |
|---|---|---|
| **Part 1** | Dry Bean classification (7 classes, 16 features, 13 611 grains) | Kernel SVMs (linear, poly, RBF, sigmoid) vs. an ANN |
| **Part 2** | Rock-Paper-Scissors images (*pierre*, *papier*, *ciseaux*) | Custom CNN (v1 → improved v2) vs. Transfer Learning (VGG16) |

### How to run it on Google Colab
1. In your Google Drive create a folder **`Test1_Hackathon`** (in *My Drive*).
2. Put inside it: **`Dry_Bean_Dataset.csv`** (or the `.xlsx` version) and the images folder **`Papier-pierre-sciseaux`** (with the 3 sub-folders `papier`, `pierre`, `sciseaux`) — or the zip file `pierre-papier-ciseaux.zip`.
3. *(Recommended)* `Runtime → Change runtime type → GPU` (T4). It makes Part 2 much faster.
4. `Runtime → Run all`. Google asks the permission to access your Drive once.

### Save / Load models on Google Drive
Every trained model (SVM, ANN, CNN, VGG16) is **saved automatically** in `Test1_Hackathon/part1_models` and `Test1_Hackathon/part2_models`.  
When you run the notebook again, **the saved models are loaded from Drive instead of being retrained** (a few seconds instead of minutes).  
→ Set `RETRAIN = True` in the config cell if you want to force a new training.

## 0. Setup

### 0.1 Imports

In [ ]:
# imbalanced-learn (SMOTE) is already on Colab most of the time, otherwise we install it
try:
    import imblearn
except ImportError:
    %pip install -q imbalanced-learn

In [ ]:
import os, glob, json, time, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from PIL import Image

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)
from imblearn.over_sampling import SMOTE

# tensorflow / keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU only - Part 2 will be slower)")

### 0.2 Google Drive, paths and configuration
On Colab the Drive is mounted; outside Colab (local Jupyter) the notebook simply uses a local folder, so the same file works in both places.

In [ ]:
try:                                                    # --- running on Google Colab
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/Test1_Hackathon"
except ImportError:                                     # --- running locally
    BASE = "./Test1_Hackathon"

P1_DIR = BASE + "/part1_models"     # SVM, ANN, scaler ... (Part 1)
P2_DIR = BASE + "/part2_models"     # CNN, VGG16 ...       (Part 2)
for d in (BASE, P1_DIR, P2_DIR):
    os.makedirs(d, exist_ok=True)

# ----------------------------- configuration -----------------------------
RETRAIN = False     # True  -> retrain everything even if a saved model exists on Drive
QUICK   = False     # True  -> tiny run (2 epochs) just to check that the notebook works
IMG_SIZE = 128      # all images are resized to 128 x 128
BATCH    = 32
EPOCHS = {"ann": 150, "cnn1": 20, "cnn2": 40, "vgg": 15, "vgg_ft": 8}   # max epochs (early stopping stops before)
if QUICK:
    EPOCHS = {k: 2 for k in EPOCHS}

print("Working folder :", os.path.abspath(BASE))
print("Files inside   :", os.listdir(BASE))

### 0.3 Helper functions (train-or-load, plots)
These small functions are used by both parts:
* `train_or_load` — if a model with this name already exists on Drive → **load it**; otherwise **train it and save it** (model + training history).
* `plot_history` — learning curves (loss and accuracy, train vs. validation).
* `plot_cm` — confusion matrix.

In [ ]:
def train_or_load(name, build_fn, Xtr, ytr, Xva, yva, folder, epochs, batch=BATCH, patience=6):
    # Keras model: load it from Drive if it exists, otherwise train + save it.
    path, hist_path = f"{folder}/{name}.keras", f"{folder}/{name}_history.json"
    if os.path.exists(path) and os.path.exists(hist_path) and not RETRAIN:
        model = keras.models.load_model(path)
        info = json.load(open(hist_path))
        print(f"[{name}] loaded from Drive  ({path})")
    else:
        model = build_fn()
        cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, patience // 2))]
        t0 = time.time()
        h = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=batch,
                      callbacks=cbs, verbose=2)
        info = {"history": {k: [float(v) for v in vs] for k, vs in h.history.items()},
                "time": time.time() - t0}
        model.save(path)
        json.dump(info, open(hist_path, "w"))
        print(f"[{name}] trained in {info['time']:.0f}s and saved to Drive  ({path})")
    return model, info


def plot_history(info, title):
    h = info["history"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h["loss"], label="train");     ax[0].plot(h["val_loss"], label="validation")
    ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="validation")
    ax[0].set_title(title + " - loss"); ax[1].set_title(title + " - accuracy")
    for a in ax:
        a.set_xlabel("epoch"); a.legend()
    plt.tight_layout(); plt.show()


def plot_cm(y_true, y_pred, names, title, ax=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    shown = cm / cm.sum(axis=1, keepdims=True) if normalize else cm
    alone = ax is None
    if alone:
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(shown, annot=True, fmt=".2f" if normalize else "d", cmap="Blues", cbar=False,
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_title(title); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    if alone:
        plt.tight_layout(); plt.show()

---
# PART 1 — Machine Learning models: Dry Bean classification

**Goal:** classify **7 varieties** of dry beans (*Seker, Barbunya, Bombay, Cali, Dermason, Horoz, Sira*) from **16 measurements** (12 dimensions + 4 shape factors) of each grain, using **Kernel SVMs** and an **ANN**, then compare them.

## 1. Step 1 — Data pre-processing
### 1.1 Import the dataset

In [ ]:
def find_file(names):
    for root, _, files in os.walk(BASE):
        for f in files:
            if f.lower() in names:
                return os.path.join(root, f)

bean_path = find_file(("dry_bean_dataset.csv",)) or find_file(("dry_bean_dataset.xlsx", "dry_bean_dataset.xls"))
print("Dataset file:", bean_path)
assert bean_path, "Dry_Bean_Dataset.csv / .xlsx not found: put it inside the Test1_Hackathon folder"

if bean_path.lower().endswith(".csv"):
    df = pd.read_csv(bean_path, sep=None, engine="python")      # sep=None -> detects , or ;
else:
    df = pd.read_excel(bean_path)
df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
display(df.head())
display(df.describe().T.round(3))

### 1.2 Missing values and duplicates
* **Missing values:** if any, numeric columns are filled with the **median** (robust to outliers); rows without a label are dropped.
* **Duplicates:** identical rows are removed — otherwise the same grain could end up in both the train and the test set (**data leakage**).

In [ ]:
print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "none")

feature_cols = [c for c in df.columns if c != "Class"]
df = df.dropna(subset=["Class"])
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicated rows removed: {n_dup}  ->  {len(df)} grains left")

### 1.3 Quick exploratory analysis
The class histogram shows the imbalance (**Dermason ≈ 3 500** grains vs. **Bombay ≈ 500**), which is exactly why the exercise insists on *balancing the dataset*.
The correlation matrix shows that many features are strongly correlated (Area, Perimeter, ConvexArea, EquivDiameter, MajorAxisLength all measure "size").

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
order = df["Class"].value_counts()
sns.barplot(x=order.index, y=order.values, ax=ax[0], palette="viridis")
for i, v in enumerate(order.values): ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[0].set_title("Number of grains per class (original data)"); ax[0].set_xlabel("")

sns.heatmap(df[feature_cols].corr(), cmap="coolwarm", center=0, ax=ax[1], cbar_kws={"shrink": .8})
ax[1].set_title("Correlation between the 16 features")
plt.tight_layout(); plt.show()

### 1.4 Encoding, splitting, scaling and **balancing**
Order of the operations (it matters!):

1. **Encode** the categorical target (`Class`: text → integers 0…6) with `LabelEncoder`.
2. **Split first**: 80 % train / 10 % validation / 10 % test, *stratified* (same class proportions in each part).  
   The validation set is used to choose the kernel / hyper-parameters and for the ANN early stopping; the test set is only used at the very end.
3. **Scale** with `StandardScaler` — **fitted on the train set only**, then applied to val and test (no leakage). SVMs and neural networks are very sensitive to feature scales.
4. **Balance the classes with SMOTE** — on the **training set only**. SMOTE creates *synthetic* grains of the minority classes (interpolation between neighbours) until all 7 classes have the same frequency.  
   The validation and test sets are left untouched (real, natural distribution), so the final scores are honest. Because the test set is still imbalanced, we also report the **macro F1-score**, which gives the same weight to every class.

In [ ]:
# 1) encoding
le = LabelEncoder().fit(df["Class"])
class_names = list(le.classes_)
X = df[feature_cols].values
y = le.transform(df["Class"])
print("Encoding:", dict(zip(class_names, range(len(class_names)))))

# 2) split 80 / 10 / 10 (stratified)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_va, X_te, y_va, y_te   = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

# 3) scaling (fit on train only)
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_va_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)

# 4) balancing with SMOTE (train only)
X_trb, y_trb = SMOTE(random_state=SEED).fit_resample(X_tr_s, y_tr)

# keep the encoder and the scaler on Drive (needed to predict new beans later)
joblib.dump(le, f"{P1_DIR}/label_encoder.joblib"); joblib.dump(scaler, f"{P1_DIR}/scaler.joblib")

print(f"\ntrain: {len(y_tr)} -> {len(y_trb)} after SMOTE | validation: {len(y_va)} | test: {len(y_te)}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].bar(class_names, np.bincount(y_tr), color="tab:red");   ax[0].set_title("Train set BEFORE balancing")
ax[1].bar(class_names, np.bincount(y_trb), color="tab:green"); ax[1].set_title("Train set AFTER SMOTE (balanced)")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

**Dimensionality check (PCA).** Before choosing the models, let's see how much *real* information the 16 features contain.

In [ ]:
pca = PCA().fit(X_tr_s)
cum = np.cumsum(pca.explained_variance_ratio_)
print(f"Components needed for 95% of the variance: {np.argmax(cum >= 0.95) + 1} / {len(cum)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, len(cum) + 1), cum, "o-"); ax[0].axhline(0.95, color="r", ls="--")
ax[0].set_xlabel("number of components"); ax[0].set_ylabel("cumulative variance"); ax[0].set_title("PCA - explained variance")
Z = pca.transform(X_tr_s)[:, :2]
for c, n in enumerate(class_names):
    ax[1].scatter(Z[y_tr == c, 0], Z[y_tr == c, 1], s=5, label=n, alpha=.6)
ax[1].legend(markerscale=3); ax[1].set_title("Train set projected on the first 2 principal components")
plt.tight_layout(); plt.show()

## 2. Step 3 — Kernel SVMs
We train the **same SVM with 4 different kernels** on the balanced training set (`C=1`, `gamma="scale"`):

| Kernel | Decision boundary |
|---|---|
| `linear` | straight hyperplanes |
| `poly` (degree 3) | polynomial surfaces |
| `rbf` | smooth, local, very flexible (Gaussian) |
| `sigmoid` | like a 1-layer neural network (tanh) |

Each model is saved to Drive (`.joblib`) and reloaded if it already exists. The **validation** score is used to compare the kernels; the **test** score is reported for the final comparison.

In [ ]:
p1_results, p1_preds = {}, {}      # scores + test predictions of every Part-1 model

def score_p1(name, model, train_time):
    pv, pt = model.predict(X_va_s), model.predict(X_te_s)
    p1_results[name] = {"Val accuracy": accuracy_score(y_va, pv), "Test accuracy": accuracy_score(y_te, pt),
                        "Test macro-F1": f1_score(y_te, pt, average="macro"), "Train time (s)": train_time}
    p1_preds[name] = pt

def svm_train_or_load(name, make_model):
    path = f"{P1_DIR}/{name}.joblib"
    if os.path.exists(path) and not RETRAIN:
        d = joblib.load(path); print(f"[{name}] loaded from Drive")
    else:
        m, t0 = make_model(), time.time()
        m.fit(X_trb, y_trb)
        d = {"model": m, "time": time.time() - t0}
        joblib.dump(d, path); print(f"[{name}] trained in {d['time']:.0f}s and saved to Drive")
    return d["model"], d["time"]

kernels = {
    "SVM linear":  lambda: SVC(kernel="linear",  C=1, random_state=SEED),
    "SVM poly":    lambda: SVC(kernel="poly", degree=3, C=1, gamma="scale", random_state=SEED),
    "SVM rbf":     lambda: SVC(kernel="rbf",     C=1, gamma="scale", random_state=SEED),
    "SVM sigmoid": lambda: SVC(kernel="sigmoid", C=1, gamma="scale", random_state=SEED),
}
svms = {}
for name, mk in kernels.items():
    svms[name], t = svm_train_or_load("svm_" + name.split()[1], mk)
    score_p1(name, svms[name], t)

display(pd.DataFrame(p1_results).T.round(4))

### 2.1 Tuning the best kernel (C and gamma)
The RBF kernel is normally the strongest one on this kind of data. We tune its two hyper-parameters with a small grid:
* **C** — penalty for misclassified points (high C = harder margin, risk of overfitting).
* **gamma** — width of the Gaussian (high gamma = very local boundary).

To keep it fast, each combination is trained on a 6 000-sample subset of the balanced train set and **scored on the validation set** (real, not synthetic data → no SMOTE leakage). The best pair is then re-trained on the full train set.

In [ ]:
tune_path = f"{P1_DIR}/svm_rbf_tuning.json"
if os.path.exists(tune_path) and not RETRAIN:
    tune = json.load(open(tune_path)); print("tuning results loaded from Drive")
else:
    sub = np.random.RandomState(SEED).choice(len(y_trb), 6000, replace=False)
    tune = []
    for C in [1, 10, 100]:
        for g in [0.01, 0.06, 0.2]:
            m = SVC(kernel="rbf", C=C, gamma=g).fit(X_trb[sub], y_trb[sub])
            tune.append({"C": C, "gamma": g, "val_acc": accuracy_score(y_va, m.predict(X_va_s))})
    json.dump(tune, open(tune_path, "w"))

tune_df = pd.DataFrame(tune)
display(tune_df.pivot(index="C", columns="gamma", values="val_acc").round(4))
best = tune_df.sort_values("val_acc", ascending=False).iloc[0]
print(f"Best: C = {best.C}, gamma = {best.gamma}")

svms["SVM rbf (tuned)"], t = svm_train_or_load("svm_rbf_tuned", lambda: SVC(kernel="rbf", C=best.C, gamma=best.gamma, random_state=SEED))
score_p1("SVM rbf (tuned)", svms["SVM rbf (tuned)"], t)
display(pd.DataFrame(p1_results).T.round(4))

In [ ]:
# Best SVM according to the VALIDATION accuracy (the test set is not used to choose)
best_svm = max(svms, key=lambda k: p1_results[k]["Val accuracy"])
print("Best SVM (validation):", best_svm)
print(classification_report(y_te, p1_preds[best_svm], target_names=class_names, digits=3))

## 3. Step 4 — ANN (Artificial Neural Network)
A fully-connected network for tabular data:

`Input(16) → Dense(128) → BatchNorm → ReLU → Dropout(0.3) → Dense(64) → BatchNorm → ReLU → Dropout(0.3) → Dense(32, ReLU) → Dense(7, softmax)`

* **softmax** output = probability of each of the 7 classes, **sparse categorical cross-entropy** loss (labels are integers), **Adam** optimizer.
* **BatchNorm + Dropout** limit overfitting; **EarlyStopping** (on the validation loss) stops the training when it stops improving and restores the best weights; **ReduceLROnPlateau** halves the learning rate when the loss stagnates.
* Trained on the balanced train set, validated on the (real) validation set.

In [ ]:
def build_ann(lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(X_trb.shape[1],)),
        layers.Dense(128), layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.3),
        layers.Dense(64),  layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(len(class_names), activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

build_ann().summary()
ann, ann_info = train_or_load("ann_dry_bean", build_ann, X_trb, y_trb, X_va_s, y_va, P1_DIR, EPOCHS["ann"], batch=64, patience=12)
plot_history(ann_info, "ANN")

In [ ]:
pv = ann.predict(X_va_s, verbose=0).argmax(1); pt = ann.predict(X_te_s, verbose=0).argmax(1)
p1_results["ANN"] = {"Val accuracy": accuracy_score(y_va, pv), "Test accuracy": accuracy_score(y_te, pt),
                     "Test macro-F1": f1_score(y_te, pt, average="macro"), "Train time (s)": ann_info["time"]}
p1_preds["ANN"] = pt
print(classification_report(y_te, pt, target_names=class_names, digits=3))

## 4. Step 5 — Comparison of the models

In [ ]:
res1 = pd.DataFrame(p1_results).T
display(res1.round(4).sort_values("Test accuracy", ascending=False))
res1.to_csv(f"{P1_DIR}/part1_results.csv")

fig, ax = plt.subplots(figsize=(11, 4))
res1[["Val accuracy", "Test accuracy", "Test macro-F1"]].plot.bar(ax=ax)
lo = res1[["Val accuracy", "Test accuracy", "Test macro-F1"]].values.min()
ax.set_ylim(max(0, lo - 0.03), 1.0); ax.set_title("Part 1 - comparison of the models"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

# confusion matrices (rows normalised: 1.00 = every grain of this class is correct)
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
plot_cm(y_te, p1_preds[best_svm], class_names, f"{best_svm} (best SVM) - test set", ax=ax[0])
plot_cm(y_te, p1_preds["ANN"],    class_names, "ANN - test set", ax=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# automatic summary written from the numbers of THIS run
best_all = res1["Test accuracy"].idxmax()
f1_cls = pd.DataFrame({m: f1_score(y_te, p1_preds[m], average=None) for m in [best_svm, "ANN"]}, index=class_names).round(3)
print(f"Best model on the test set : {best_all}  ->  accuracy {res1.loc[best_all, 'Test accuracy']:.2%}, macro-F1 {res1.loc[best_all, 'Test macro-F1']:.3f}")
print(f"Best SVM kernel            : {best_svm}")
print(f"ANN                        : accuracy {res1.loc['ANN', 'Test accuracy']:.2%}, macro-F1 {res1.loc['ANN', 'Test macro-F1']:.3f}")
print("\nF1-score per class (test set):"); display(f1_cls)

### Discussion of the results (Part 1)
*(The numbers below come from our run; on your machine they can move by a few tenths of a point, the table above is the reference.)*

**1. What the numbers say.** The RBF SVM (≈ 92 % accuracy, macro-F1 ≈ 0.93) and the ANN (≈ 92 %, macro-F1 ≈ 0.93) end up **practically tied**; the linear SVM is only ≈ 1 point lower, the degree-3 polynomial SVM ≈ 2 points lower, and the sigmoid SVM is far behind (≈ 69 %). The gap between the best SVM and the ANN is a handful of grains out of ≈ 1 350 test grains, while the standard error of an accuracy of 92 % on a test set of this size is ≈ 0.7 point — so **the ANN is not significantly better or worse than the RBF SVM**. The honest conclusion is a tie on accuracy, and a clear difference in *cost*: the SVM trains in a few seconds, the ANN needs about a minute of epochs, early stopping and tuning of the architecture.

**2. Why the ANN does not beat the SVM here — dimensionality and characteristics of the data.**
* **Low dimension, medium size.** 16 features and ≈ 13 500 samples is the comfort zone of kernel SVMs. The strength of neural networks is to *learn* hierarchical representations from raw, high-dimensional data (pixels, audio, text); with 16 already-engineered measurements there is almost nothing left to learn, so the extra capacity of the ANN brings no gain.
* **Highly redundant features.** All 16 attributes are geometric functions of the same bean silhouette (Area, Perimeter, ConvexArea, EquivDiameter, MajorAxisLength… all measure *size*). The PCA shows that **≈ 4 components keep 95 % of the variance**: the real dimension of the problem is small, and both models find the same smooth boundary in that small space.
* **Nearly linearly separable classes.** The *linear* SVM already reaches ≈ 91 %; the RBF kernel only adds ≈ 1 point by bending the boundary a little. This is why a more flexible model has little room to improve.
* **Why the other kernels are worse.** The degree-3 polynomial kernel is too flexible/unstable on standardised features that contain outliers (its values explode with the degree). The **sigmoid kernel** is not a valid positive-semi-definite kernel in general and is very sensitive to `gamma` and `coef0`; with the default values it fails — this says nothing about the data, only that this kernel was not tuned.
* **Tuning does not always help.** The tuned RBF (best (C, gamma) of the grid on the validation set) is *not* better than the default `gamma="scale"` on the test set (≈ 91.4 % vs ≈ 92.2 %). The differences are within the noise, which is a reminder not to over-tune on a validation set of ≈ 1 350 samples.

**3. Where the errors are.** *Bombay* is perfectly classified (F1 = 1.00): these beans are much bigger than the others. Most mistakes are between **Sira, Dermason, Seker, Barbunya and Cali** (*Sira* has the lowest F1 ≈ 0.87): these varieties have **very similar sizes and shapes**, and the dataset contains *only shape measurements* (no colour, no texture). This sets a **ceiling of ≈ 92–93 %** for any model trained on these 16 features — to go higher we would need *new information* (colour/texture features, or the images themselves with a CNN), not a more complex classifier.

**4. Effect of the balancing.** Without balancing, the classifiers would be pushed towards the big classes (Dermason ≈ 3 500 grains vs. Bombay ≈ 500). With SMOTE on the training set, every variety has the same weight, and the macro-F1 (which weights every class equally) is **higher than the accuracy** — the small classes are well recognised. The test set was kept in its natural distribution, so the scores are not inflated.

**5. Practical conclusion.** On this dataset the **RBF-kernel SVM is the best choice**: best (or tied-best) accuracy on validation *and* test, almost no tuning, trained in seconds, easy to save/reload. The ANN is a valid alternative (it directly outputs class probabilities and scales better if the dataset becomes much bigger), but it is not worth its extra complexity here.

## 5. Save / Load demo — predicting a new bean from the saved models
In a **new Colab session** you do not need to train anything: load the encoder, the scaler and the models from Drive and predict.

In [ ]:
le_l     = joblib.load(f"{P1_DIR}/label_encoder.joblib")
scaler_l = joblib.load(f"{P1_DIR}/scaler.joblib")
svm_l    = joblib.load(f"{P1_DIR}/svm_rbf_tuned.joblib")["model"]
ann_l    = keras.models.load_model(f"{P1_DIR}/ann_dry_bean.keras")

def predict_bean(raw_features):
    # raw_features: the 16 measurements of ONE bean (unscaled), in the same order as the dataset
    z = scaler_l.transform(np.array(raw_features, dtype=float).reshape(1, -1))
    p_svm = le_l.inverse_transform(svm_l.predict(z))[0]
    proba = ann_l.predict(z, verbose=0)[0]
    return p_svm, le_l.classes_[proba.argmax()], proba.max()

for i in range(5):
    svm_p, ann_p, conf = predict_bean(X_te[i])
    print(f"true: {class_names[y_te[i]]:9s} | SVM: {svm_p:9s} | ANN: {ann_p:9s} ({conf:.0%})")

---
# PART 2 — CNN & Transfer Learning: Rock-Paper-Scissors

**Goal:** classify hand images into **3 classes** — *pierre* (rock), *papier* (paper), *ciseaux* (scissors) — with
1. a **CNN built by us** (first version v1, then an improved version v2),
2. a **CNN with Transfer Learning** (VGG16 pre-trained on ImageNet, frozen layers + our own classification head),

and then compare them.

## 6. Step 1 — Data pre-processing
### 6.1 Load the images
The notebook looks for the folder that contains the 3 class sub-folders (`papier`, `pierre`, `sciseaux`) anywhere inside `Test1_Hackathon`; if the dataset is a **zip** file, it is extracted first. Every image is converted to RGB and resized to 128×128.  
*(If the automatic search fails, write the path by hand in `DATA_DIR`.)*

In [ ]:
DATA_DIR = None        # <- optional: write the folder path by hand, e.g. BASE + "/Papier-pierre-sciseaux"
ALIASES = {"papier", "pierre", "sciseaux", "ciseaux", "paper", "rock", "scissors"}

def find_data_dir():
    for root in [BASE, "/tmp/rps"]:
        for dp, dn, _ in os.walk(root):
            if "models" in dp: continue
            if sum(d.lower() in ALIASES for d in dn) == 3:
                return dp

if DATA_DIR is None:
    DATA_DIR = find_data_dir()
    if DATA_DIR is None:                                   # not found -> maybe it is a zip file
        for z in glob.glob(BASE + "/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/tmp/rps")
        DATA_DIR = find_data_dir()
assert DATA_DIR, "Images folder not found: put 'Papier-pierre-sciseaux' (or the zip) inside Test1_Hackathon"
print("Dataset folder:", DATA_DIR)

class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print("Classes:", class_names)

images, labels = [], []
for label, name in enumerate(class_names):
    folder = os.path.join(DATA_DIR, name)
    for f in sorted(os.listdir(folder)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            try:
                img = Image.open(os.path.join(folder, f)).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                images.append(np.array(img)); labels.append(label)
            except Exception:
                print("unreadable file skipped:", f)
X = np.array(images, dtype="uint8"); y = np.array(labels)
del images
print("Images:", X.shape, "| labels:", y.shape)

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i, name in enumerate(class_names):
    idx = np.random.RandomState(SEED).choice(np.where(y == i)[0], 6, replace=False)
    for j, k in enumerate(idx):
        axes[i, j].imshow(X[k]); axes[i, j].axis("off")
    axes[i, 0].set_title(name, loc="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

### 6.2 Histogram of the class frequencies

In [ ]:
counts = np.bincount(y)
plt.figure(figsize=(6, 3.5))
plt.bar(class_names, counts, color=["tab:blue", "tab:orange", "tab:green"])
for i, c in enumerate(counts): plt.text(i, c, str(c), ha="center", va="bottom")
plt.title("Number of images per class (whole dataset)"); plt.ylabel("images"); plt.show()
print(dict(zip(class_names, counts)))

### 6.3 Train / validation / test split
The sheet says *"80 % pour test et 20 % pour la validation"*. This is almost certainly a **typo** (it should be 80 % for the **train** set). So the split used here is:

* **80 % train** — to fit the model,
* **10 % validation** — to tune the hyper-parameters and for early stopping,
* **10 % test** — final evaluation, never used for tuning.

(The 20 % that is not used for training is simply cut in two halves, so both readings of "20 %" are covered.)  
The split is **stratified** and done **before** the augmentation, so an augmented copy of a training image can never appear in the validation/test sets (no leakage).

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test   = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
del X
print(f"train: {len(y_train)} | validation: {len(y_val)} | test: {len(y_test)}")

### 6.4 Image augmentation — two goals
**(a) Balance the classes:** new images are generated from the *training* images of the smaller classes (random flip, rotation, zoom, shift, contrast) until every class has as many images as the biggest one.  
**(b) Create variability:** the same augmentation block is also placed **inside** the improved CNN and the VGG16 model, so at every epoch the network sees slightly different images (active only during training, switched off at prediction).

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.15),
], name="augmentation")

# (a) balance the train set with augmented copies of the minority classes
max_count = np.bincount(y_train).max()
new_X, new_y = [], []
for c in range(len(class_names)):
    imgs_c = X_train[y_train == c]
    missing = max_count - len(imgs_c)
    if missing > 0:
        picks = imgs_c[np.random.RandomState(SEED + c).choice(len(imgs_c), missing)]
        aug = augment(picks.astype("float32"), training=True).numpy().clip(0, 255).astype("uint8")
        new_X.append(aug); new_y += [c] * missing

if new_X:
    X_train_bal = np.concatenate([X_train] + new_X); y_train_bal = np.concatenate([y_train, np.array(new_y)])
else:
    X_train_bal, y_train_bal = X_train, y_train
print(f"train set: {len(y_train)} -> {len(y_train_bal)} images after balancing")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(class_names, np.bincount(y_train)); ax[0].set_title("Train set BEFORE augmentation")
ax[1].bar(class_names, np.bincount(y_train_bal), color="tab:green"); ax[1].set_title("Train set AFTER augmentation")
plt.tight_layout(); plt.show()

# (b) what the augmentation does to one image
img = X_train[0:1].astype("float32")
fig, axes = plt.subplots(1, 7, figsize=(16, 2.8))
axes[0].imshow(X_train[0]); axes[0].set_title("original")
for i in range(1, 7):
    axes[i].imshow(augment(img, training=True).numpy()[0].clip(0, 255).astype("uint8")); axes[i].set_title(f"augmented {i}")
for a in axes: a.axis("off")
plt.show()

### 6.5 Evaluation helper
Used for the three models to compute the scores and to draw the report / confusion matrix.

In [ ]:
# custom layer = same preprocessing as VGG16 (RGB -> BGR, subtract the ImageNet mean). Registered so that .keras files can be reloaded.
@keras.utils.register_keras_serializable(package="rps")
class VGGPreprocess(layers.Layer):
    def call(self, x):
        x = tf.cast(x, tf.float32)
        x = tf.reverse(x, axis=[-1])
        return x - tf.constant([103.939, 116.779, 123.68])

p2_results, p2_probas = {}, {}

def evaluate_p2(label, model, info):
    t0 = time.time(); p = model.predict(X_test, verbose=0); inf_ms = (time.time() - t0) / len(X_test) * 1000
    pred = p.argmax(1); p2_probas[label] = p
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    p2_results[label] = {"Val accuracy": accuracy_score(y_val, model.predict(X_val, verbose=0).argmax(1)),
                         "Test accuracy": accuracy_score(y_test, pred),
                         "Test macro-F1": f1_score(y_test, pred, average="macro"),
                         "Total params": model.count_params(), "Trainable params": trainable,
                         "Epochs": len(info["history"]["loss"]), "Train time (s)": info["time"],
                         "Inference (ms/img)": inf_ms}
    print(classification_report(y_test, pred, target_names=class_names, digits=3))
    plot_cm(y_test, pred, class_names, label + " - confusion matrix (test)", normalize=False)

## 7. Step 2 — Custom CNN (version 1)
`3 × [Conv2D + MaxPooling] → Flatten → Dense(128) → Dropout → Dense(3, softmax)`

* `Rescaling(1/255)` puts the pixels in [0, 1].
* Compiled with **Adam** and **sparse categorical cross-entropy** (integer labels, 3 classes → softmax).

In [ ]:
def build_cnn_v1(lr=1e-3, dropout=0.5):
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Rescaling(1 / 255.),
        layers.Conv2D(32, 3, activation="relu"),  layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),  layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(len(class_names), activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

build_cnn_v1().summary()

## 8. Step 3 — Training and hyper-parameter tuning
The **validation set** is used to choose the **learning rate** and the **dropout rate**: 4 combinations are trained for 5 epochs each and we keep the best validation accuracy. (The results are saved on Drive too.)

In [ ]:
tune_path = P2_DIR + "/cnn_tuning.json"
if os.path.exists(tune_path) and not RETRAIN:
    tuning = json.load(open(tune_path)); print("tuning results loaded from Drive")
else:
    tuning = []
    for lr in [1e-3, 3e-4]:
        for dp in [0.3, 0.5]:
            m = build_cnn_v1(lr, dp)
            h = m.fit(X_train_bal, y_train_bal, validation_data=(X_val, y_val), epochs=1 if QUICK else 5, batch_size=BATCH, verbose=0)
            tuning.append({"lr": lr, "dropout": dp, "val_acc": max(h.history["val_accuracy"])})
    json.dump(tuning, open(tune_path, "w"))

tuning_df = pd.DataFrame(tuning).sort_values("val_acc", ascending=False)
display(tuning_df.round(4))
best_lr, best_dp = float(tuning_df.iloc[0]["lr"]), float(tuning_df.iloc[0]["dropout"])
print("Best combination -> lr =", best_lr, "| dropout =", best_dp)

In [ ]:
cnn1, info1 = train_or_load("cnn_v1", lambda: build_cnn_v1(best_lr, best_dp), X_train_bal, y_train_bal, X_val, y_val, P2_DIR, EPOCHS["cnn1"])
plot_history(info1, "CNN v1")

## 9. Step 4 — Evaluation (test set)

In [ ]:
evaluate_p2("CNN v1", cnn1, info1)

### 9.1 Improving the architecture → CNN v2
After looking at the curves of v1 (training accuracy higher than validation accuracy = **overfitting**, and a `Flatten → Dense` layer with millions of weights), these changes are made:

| Change | Why |
|---|---|
| Augmentation layers **inside** the model | different images at each epoch → less overfitting, more robust model |
| 4 blocks of **2 × Conv2D** (32-64-128-256 filters) | deeper network → richer features (edges → fingers → hand shape) |
| **BatchNormalization** after each convolution | more stable and faster training, allows a higher learning rate |
| **GlobalAveragePooling** instead of Flatten | far fewer parameters in the dense part → less overfitting |
| **Dropout 0.25** in the conv blocks + dropout in the head | regularization |
| EarlyStopping + ReduceLROnPlateau | stop at the best epoch, finer learning at the end |

In [ ]:
def build_cnn_v2(lr=1e-3, dropout=0.5):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augment(inputs)                          # only active during training
    x = layers.Rescaling(1 / 255.)(x)
    for f in [32, 64, 128, 256]:
        for _ in range(2):
            x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

cnn2, info2 = train_or_load("cnn_v2", lambda: build_cnn_v2(best_lr, best_dp), X_train_bal, y_train_bal, X_val, y_val, P2_DIR, EPOCHS["cnn2"], patience=8)
plot_history(info2, "CNN v2 (improved)")
evaluate_p2("CNN v2 (improved)", cnn2, info2)

## 10. Step 5 — CNN with Transfer Learning (VGG16)
**Transfer learning** = re-use a network that was already trained on a huge dataset (ImageNet: 1.2 million images, 1000 classes) for our small task.

1. Load **VGG16** *without* its original classifier (`include_top=False`), with the ImageNet weights.
2. **Freeze** all its layers (`trainable = False`) → the pre-trained knowledge is not destroyed.
3. Add **our own layers**: `GlobalAveragePooling → Dense(256, ReLU) → Dropout(0.5) → Dense(3, softmax)`.
4. **Compile** with the **Adam** optimizer and the **sparse categorical cross-entropy** loss. Only our new layers are trained.

In [ ]:
def build_vgg(lr=1e-3):
    base = keras.applications.VGG16(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False                            # <- frozen pre-trained layers
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augment(inputs)
    x = VGGPreprocess()(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

vgg, info3 = train_or_load("vgg16_frozen", build_vgg, X_train_bal, y_train_bal, X_val, y_val, P2_DIR, EPOCHS["vgg"], patience=4)
n_train = int(sum(np.prod(w.shape) for w in vgg.trainable_weights))
print(f"Total parameters: {vgg.count_params():,} | trainable (our head only): {n_train:,}")
plot_history(info3, "VGG16 (frozen)")
evaluate_p2("VGG16 (transfer learning)", vgg, info3)

### 10.1 *(Bonus)* Fine-tuning the last block of VGG16
Once the new head is trained, we can **unfreeze the last convolution block (`block5`)** and continue the training with a very small learning rate (`1e-5`), so VGG16 adapts its high-level filters to hands. Set `RUN_FINE_TUNING = False` to skip this bonus.

In [ ]:
RUN_FINE_TUNING = True

def build_vgg_finetune():
    m = keras.models.load_model(f"{P2_DIR}/vgg16_frozen.keras")      # start from the trained frozen model
    base = m.get_layer("vgg16")
    base.trainable = True
    for l in base.layers:
        l.trainable = l.name.startswith("block5")
    m.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

if RUN_FINE_TUNING:
    vgg_ft, info4 = train_or_load("vgg16_finetuned", build_vgg_finetune, X_train_bal, y_train_bal, X_val, y_val, P2_DIR, EPOCHS["vgg_ft"], patience=3)
    plot_history(info4, "VGG16 fine-tuned")
    evaluate_p2("VGG16 (fine-tuned, bonus)", vgg_ft, info4)

## 11. Step 6 — Comparison and discussion

In [ ]:
res2 = pd.DataFrame(p2_results).T
display(res2.round(4))
res2.to_csv(f"{P2_DIR}/part2_results.csv")

res2[["Val accuracy", "Test accuracy", "Test macro-F1"]].plot.bar(figsize=(9, 4))
plt.ylim(max(0, res2[["Val accuracy", "Test accuracy"]].values.min() - 0.05), 1.01)
plt.xticks(rotation=15); plt.title("Part 2 - comparison of the models"); plt.tight_layout(); plt.show()

In [ ]:
# the images that the best model gets wrong
best2 = res2["Test accuracy"].idxmax()
wrong = np.where(p2_probas[best2].argmax(1) != y_test)[0]
print(f"{best2}: {len(wrong)} wrong predictions out of {len(y_test)} test images")
if len(wrong):
    n = min(8, len(wrong)); fig, axes = plt.subplots(1, n, figsize=(2.4 * n, 3))
    for a, k in zip(np.atleast_1d(axes), wrong[:n]):
        a.imshow(X_test[k]); a.axis("off")
        a.set_title(f"true: {class_names[y_test[k]]}\npred: {class_names[p2_probas[best2][k].argmax()]}", fontsize=9)
    plt.show()

# short summary written from the numbers of THIS run
print("\nBest model on the test set:", best2, "->", f"{res2.loc[best2, 'Test accuracy']:.2%}")
for m_ in res2.index:
    r = res2.loc[m_]
    print(f"{m_:28s} acc {r['Test accuracy']:.2%} | trainable params {int(r['Trainable params']):>10,} | epochs {int(r['Epochs']):>2} | time {r['Train time (s)']:.0f}s")

### Discussion of the results (Part 2)
*(Read this together with the results table, the learning curves and the automatic summary printed above — they contain the exact numbers of your run.)*

**1. CNN v1 → CNN v2 (improvement of our own architecture).** The learning curves of v1 typically show a training accuracy that keeps rising while the validation accuracy stops improving: **overfitting**, caused by the large `Flatten → Dense` layer (millions of weights) and by the absence of regularisation. The v2 changes attack exactly this: in-model augmentation (different images at each epoch), BatchNormalization, GlobalAveragePooling (far fewer weights in the dense part), dropout, and early stopping. The result to look for in the table is a **smaller gap between train and validation curves** and a higher validation/test accuracy for v2.

**2. Efficiency of transfer learning.** VGG16 was pre-trained on 1.2 million ImageNet images, so its convolution layers already detect edges, textures, contours and shapes. We keep them **frozen** and train only our small head (**≈ 132 000 trainable parameters out of ≈ 14.8 million**). The consequences are visible in the table:
* **Fast convergence** — a good accuracy after very few epochs, with little hyper-parameter tuning;
* **Less overfitting and less data needed** — only a tiny number of weights is learned from our images, so the model generalises well even with a few thousand pictures;
* **Strong accuracy** — the features come from a network trained on far more data than we have.

**3. Comparison with our CNN.** Transfer learning is expected to be at least as accurate as the best custom CNN, with a much smaller number of *trainable* parameters and fewer epochs. A custom CNN trained from scratch must learn *everything* (edges → fingers → hand shape) from ≈ 4 000 images, which is why it needs augmentation, BatchNorm and a careful architecture to get close. The rock-paper-scissors task is easy (3 very different hand shapes, uniform background), so **both families can reach a very high accuracy**; if the scores are close, the decision is made on the other criteria.

**4. Limits and trade-offs.**
* VGG16 is **heavy** (≈ 15 M parameters, ≈ 60 MB, slower prediction — see the *Inference (ms/img)* column), whereas our custom CNN is much smaller and faster: a better candidate for an **embedded / mobile system** (our SIEC field).
* ImageNet photos are not hand images with a plain background, and the **frozen** base cannot adapt to them. The bonus **fine-tuning of `block5`** (very small learning rate) lets VGG16 adapt its last filters; it can give a small extra gain, at the price of more computation and a risk of overfitting if the learning rate is too high.
* Augmentation and class balancing were applied to the **training set only** (the split was done first), so the validation and test scores are not contaminated by augmented copies of the same image.

**5. Conclusion.** Transfer learning is the most **efficient** approach (best accuracy-per-training-effort: pre-trained knowledge, few trainable weights, quick training). The improved custom CNN is the best choice when a **small and fast** model is required. In both cases the main levers were the same ones: balanced training data, augmentation, regularisation and monitoring the validation set.

## 12. Save / Load demo — predicting a new image from the saved model
Nothing is trained here: the model is simply **loaded from Google Drive**. Put any image in the folder and call `predict_image(path)`.

In [ ]:
loaded = keras.models.load_model(f"{P2_DIR}/vgg16_frozen.keras")      # <- load from Drive
print("Test accuracy of the loaded model:", round(accuracy_score(y_test, loaded.predict(X_test, verbose=0).argmax(1)), 4))

def predict_image(path, model=loaded):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    p = model.predict(np.array(img, dtype="float32")[None], verbose=0)[0]
    plt.imshow(img); plt.axis("off"); plt.title(f"{class_names[p.argmax()]}  ({p.max():.0%})"); plt.show()

# example with 4 images of the test set
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
pr = loaded.predict(X_test[:4], verbose=0)
for i, a in enumerate(axes):
    a.imshow(X_test[i]); a.axis("off")
    a.set_title(f"true: {class_names[y_test[i]]}\npred: {class_names[pr[i].argmax()]} ({pr[i].max():.0%})", fontsize=10)
plt.show()

---
# Final conclusion
* **Part 1 (tabular data, 16 features):** see the discussion of section 4 — all the reasonable models reach nearly the same level, the limit being the *information in the features* (Dermason / Sira / Seker look alike), not the algorithm.
* **Part 2 (images):** see the discussion of section 11 — transfer learning gives a strong model quickly because the convolution filters were already learned on ImageNet.
* **Good practices used everywhere:** split *before* balancing/augmenting, scaler fitted on train only, validation set for tuning, test set only at the end, fixed random seed, every model **saved on Google Drive and reloadable**.

*Ahmed Nadir — M2 SIEC — Université de Ghardaïa*